In [1]:
from MIDI_AUTOMATION.constants import voice_struct, VOICE_KEYS, checksum

def reorder_operator_block(block):
    # block: length 21
    block = block[:] + [0]
    new_order = [
        0, 1, 2, 3,    # R1-R4
        4, 5, 6, 7,    # L1-L4
        8, 9, 10,      # BP, LD, RD
        12, 11,        # RC, LC
        20, 13,        # DET, RS
        15, 14,        # KVS, AMS
        16,            # OL
        18, 17,        # FC, M
        19             # FF
    ]

    block = block[new_order]
    block[13] = 0
    return block

def transform_params(arr):
    arr = np.asarray(arr)
    operators = arr[:126].reshape(6, 21)  # 6*21=126
    tail = arr[126:]  # remaining global params

    new_order = [
        0, 1, 2, 3,    # PG1-PG4
        4, 5, 6, 7,    # PL1-PL4
        8, 10, 9,      # ALG, OSC, FB
        11, 12,        # LFS, LFD
        13, 14,        # LPMD, LAMD
        17, 16, 15,    # LPMS, LFW, LKS
        16            # TRNSP
    ]

    tail = tail[new_order]
    tail = np.append(tail, [0] * 10)

    reordered_ops = np.array([reorder_operator_block(op) for op in operators])
    return np.concatenate([reordered_ops.flatten(), tail])


def dx7_pack(voices):
    HEADER = int('0x43', 0), int('0x00', 0), int('0x09', 0), int('0x20', 0), int('0x00', 0)

    voices_bytes = bytes()
    for voice in voices:
        voice = transform_params(voice)
        voice_bytes = voice_struct.pack(dict(zip(VOICE_KEYS, voice)))
        voices_bytes += voice_bytes

    patch_checksum = [checksum(voices_bytes)]

    data = bytes(HEADER) + voices_bytes + bytes(patch_checksum)

    return mido.Message('sysex', data=data)

def undo_one_hot(processor, df_encoded):
    df_out = {}
    idx = 0

    for p in processor.params:

        if p.p_type == 'm':
            df_out[p.name] = df_encoded.iloc[:, idx]
            idx += 1

        elif p.p_type == 'b':
            cols = df_encoded.iloc[:, idx:idx+2].to_numpy()
            df_out[p.name] = cols.argmax(axis=1)
            idx += 2

        elif p.p_type in ('c', 'x'):
            cols = df_encoded.iloc[:, idx:idx+p.n_classes]
            df_out[p.name] = cols.values.argmax(axis=1)
            idx += p.n_classes

    return pd.DataFrame(df_out)

def denormalize(processor, df):
    df = df.copy()

    for p in processor.params:
        if p.p_type == 'm':
            df[p.name] = df[p.name] * p.p_max

    return df

def cast_patch_to_int(processor, df):
    df = df.copy()

    for p in processor.params:
        if p.p_type == 'm':
            df[p.name] = df[p.name].round().astype(int)
        else:
            df[p.name] = df[p.name].astype(int)

    df[df < 0] = 0

    return df

In [ ]:
import pretty_midi
import random

def generate_random_midi(path):
    midi = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)

    t = 0
    for _ in range(random.randint(6,12)):
        pitch = random.randint(36,84)
        dur = random.uniform(0.2,1.2)

        note = pretty_midi.Note(
            velocity=random.randint(60,120),
            pitch=pitch,
            start=t,
            end=t+dur
        )

        inst.notes.append(note)
        t += dur

    midi.instruments.append(inst)
    midi.write(path)

In [ ]:
mins = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

maxs = [99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 31, 7, 1, 99, 99, 99, 99, 1, 5, 7, 48]

In [4]:
import dawdreamer as daw
import pathlib

# --- Configuration — point this to your Dexed VST ------------------
DEXED_PATH = r"C:\Program Files\Common Files\VST3\Dexed.vst3"
OUT_DIR = pathlib.Path("./states")
OUT_DIR.mkdir(exist_ok=True)

SAMPLE_RATE = 44100
BUFFER_SIZE = 128

# Example mapping function
def normalize_value(val, mn, mx):
    """
    Normalize a raw parameter value into [0,1].
    Clip if outside bounds.
    """
    # clamp
    if val < mn:
        val = mn
    if val > mx:
        val = mx

    # normalize
    return (val - mn) / (mx - mn) if (mx - mn) > 0 else 0.0

def patch_to_state(patch, mins, maxs, name):
    """
    patch: list/array of raw parameter values
    mins: corresponding minimums
    maxs: corresponding maximums
    name: base name for the saved state
    """
    # make the audio engine
    engine = daw.RenderEngine(SAMPLE_RATE, BUFFER_SIZE)

    # create the Dexed instance
    synth = engine.make_plugin_processor("dexed", DEXED_PATH)

    # build processing graph (just the synth)
    engine.load_graph([(synth, [])])

    # Set all parameters by index
    for idx, raw_value in enumerate(patch):
        # normalize
        norm = normalize_value(raw_value, mins[idx], maxs[idx])
        synth.set_parameter(idx, norm)

    # Save plugin state file
    state_file = OUT_DIR / f"{name}.state"
    synth.save_state(str(state_file))

    print(f"Saved state: {state_file}")

    return state_file

In [ ]:
import numpy as np
import mido
import pandas as pd

df = pd.read_csv("E:\\Coding\\vae-main\\dx7\\dx7.patchparams")

first = True
index = 0
std = 2
iters = 8

for iters in range(iters):
    for r in df.iterrows():
        values = [x for x in r[1]][1:]

        # noise = np.random.normal(0, std, size=len(values)).astype(int)
        # noisy = values + noise
        # noisy = np.clip(noisy, mins, maxs)

        name = f"{index}"
        index += 1
        patch_to_state(values, mins, maxs, name)

Saved state: states\0.state
Saved state: states\1.state
Saved state: states\2.state
Saved state: states\3.state
Saved state: states\4.state
Saved state: states\5.state
Saved state: states\6.state
Saved state: states\7.state
Saved state: states\8.state
Saved state: states\9.state
Saved state: states\10.state
Saved state: states\11.state
Saved state: states\12.state
Saved state: states\13.state
Saved state: states\14.state
Saved state: states\15.state
Saved state: states\16.state
Saved state: states\17.state
Saved state: states\18.state
Saved state: states\19.state
Saved state: states\20.state
Saved state: states\21.state
Saved state: states\22.state
Saved state: states\23.state
Saved state: states\24.state
Saved state: states\25.state
Saved state: states\26.state
Saved state: states\27.state
Saved state: states\28.state
Saved state: states\29.state
Saved state: states\30.state
Saved state: states\31.state
Saved state: states\32.state
Saved state: states\33.state
Saved state: states\34.s

In [5]:
num_midi_files = 2 ** 15
for index in range(num_midi_files):
    generate_random_midi(f"./midi/{index:06d}.midi")

In [6]:
import numpy as np

spec = np.load("E:\\Coding\\vae-main\\MIDI_AUTOMATION\\npy\\patch_0000001-013777.npy")

In [4]:
spec

array([[-23.47924  , -14.245521 , -11.607346 , ..., -12.959444 ,
        -13.367077 , -15.341852 ],
       [-14.14889  , -10.361845 ,  -7.403447 , ...,  -9.929436 ,
         -8.139163 , -14.057306 ],
       [-13.252167 ,  -8.95838  ,  -6.125002 , ...,  -9.405944 ,
         -6.9072895, -11.168172 ],
       ...,
       [-80.       , -80.       , -79.86358  , ..., -80.       ,
        -80.       , -65.98436  ],
       [-80.       , -80.       , -80.       , ..., -80.       ,
        -80.       , -65.899185 ],
       [-80.       , -80.       , -80.       , ..., -80.       ,
        -80.       , -65.87255  ]], shape=(128, 345), dtype=float32)